# Model Training

In [3]:
import importlib

from torch.utils.data import DataLoader

from Config import constants
from Config import model_configs
import torch
import random
import numpy as np
import pandas as pd
from src.preprocessing.GTSRBDataset import GTSRBDataset
from src.preprocessing.transformations import get_transforms
from src.models.experiment_manager import get_experiment_setup
from src.models.model_factory import create_model
from src.models.optimizer_factory import get_optimizer,get_criterion

importlib.reload(constants)
importlib.reload(model_configs)

<module 'Config.model_configs' from 'F:\\Software\\Dataspell\\TrafficSigns\\Config\\model_configs.py'>

## Set Seed

In [4]:
torch.manual_seed(constants.SEED)
random.seed(constants.SEED)
np.random.seed(constants.SEED)

## GPU

In [5]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
NVIDIA GeForce RTX 3070


## Model Config

In [9]:
cfg_name = "exp001_b0_frozen_head"

In [10]:
cfg = model_configs.CONFIGS[cfg_name]

IMAGE_SIZE = cfg["image_size"]
BATCH_SIZE = cfg["batch_size"]
lr = cfg["lr"]

## Train Test Split

In [11]:
train_df = pd.read_csv(constants.TRAIN_CSV_PATH)
val_df = pd.read_csv(constants.VAL_CSV_PATH)
test_df = pd.read_csv(constants.TEST_CSV_PATH)

In [12]:
train_ds = GTSRBDataset(train_df, transform=get_transforms(IMAGE_SIZE))
val_ds = GTSRBDataset(val_df, transform=get_transforms(IMAGE_SIZE, train=False))
test_ds = GTSRBDataset(val_df, transform=get_transforms(IMAGE_SIZE, train=False))

In [13]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

## Model

In [15]:
experiment_setup = get_experiment_setup(cfg, num_classes=constants.NUM_CLASSES)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

F:\Software\Dataspell\TrafficSigns\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\danilo\.cache\huggingface\hub\models--timm--efficientnet_b0.ra_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [19]:
experiment_setup["cfg"]

{'model_name': 'efficientnet_b0',
 'image_size': 224,
 'batch_size': 64,
 'lr': 0.001,
 'freeze_backbone': True,
 'unfreeze_last_n': 1,
 'optimizer_name': 'adam',
 'loss_name': 'cross_entropy',
 'dropout': 0.3,
 'weight_decay': 0.0001}